# Crop Recommendation - Crop Prediction
**Role:** Jaddu Gowthami — Crop Prediction

This notebook loads the Logistic Regression model, scaler, and label encoder trained in the **Logistic Regression Model notebook** (`crop_logistic_model.pkl`, `crop_scaler.pkl`, `crop_label_encoder.pkl`, `crop_feature_columns.pkl`) and uses them to recommend the best crop for given soil and climate conditions.

> Run the Logistic Regression notebook first (or place those 4 `.pkl` files in the same folder as this notebook) before running the cells below.


In [ ]:
import pandas as pd
import numpy as np
import joblib


## 1. Load Trained Model Artifacts


In [ ]:
log_model = joblib.load('crop_logistic_model.pkl')
scaler = joblib.load('crop_scaler.pkl')
le = joblib.load('crop_label_encoder.pkl')
columns = joblib.load('crop_feature_columns.pkl')

print("Model, scaler, and label encoder loaded successfully!")
print("Number of crop classes:", len(le.classes_))
print("Classes:", list(le.classes_))

## 10. Crop Prediction Function
Implements the actual prediction task: given soil & climate parameters, recommend the best crop.

In [14]:
def predict_crop(N, P, K, temperature, humidity, ph, rainfall):
    """Predict the best crop given soil nutrient and climate parameters."""
    input_df = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]], columns=columns)
    input_scaled = scaler.transform(input_df)

    pred_encoded = log_model.predict(input_scaled)[0]
    pred_crop = le.inverse_transform([pred_encoded])[0]

    # Top-3 most probable crops, for extra context
    probs = log_model.predict_proba(input_scaled)[0]
    top3_idx = np.argsort(probs)[-3:][::-1]
    top3 = [(le.inverse_transform([i])[0], round(probs[i]*100, 2)) for i in top3_idx]

    return pred_crop, top3

In [15]:
# Example usage
result, top3 = predict_crop(N=90, P=42, K=43, temperature=20.8, humidity=82, ph=6.5, rainfall=202.9)

print("Recommended Crop:", result)
print("Top 3 probable crops:", top3)

Recommended Crop: rice
Top 3 probable crops: [('rice', np.float64(83.15)), ('jute', np.float64(15.85)), ('coffee', np.float64(0.54))]


## 3. More Example Predictions
A few more soil/climate combinations to sanity-check the prediction function.


In [ ]:
examples = [
    dict(N=20, P=134, K=200, temperature=22.6, humidity=91.5, ph=5.9, rainfall=112.6),   # expect: apple-ish
    dict(N=118, P=32, K=34, temperature=25.6, humidity=80.3, ph=6.8, rainfall=243.0),    # expect: rice-ish
    dict(N=4, P=17, K=17, temperature=28.6, humidity=27.5, ph=7.0, rainfall=63.3),       # expect: chickpea-ish
]

for ex in examples:
    crop, top3 = predict_crop(**ex)
    print(f"Input: {ex}")
    print(f"  Recommended Crop: {crop}")
    print(f"  Top 3: {top3}")
    print()

## 4. Interactive Prediction
Run this cell and enter your own soil/climate values to get a live crop recommendation.


In [ ]:
def predict_crop_interactive():
    print("Enter soil & climate parameters:")
    N = float(input("Nitrogen (N): "))
    P = float(input("Phosphorus (P): "))
    K = float(input("Potassium (K): "))
    temperature = float(input("Temperature (°C): "))
    humidity = float(input("Humidity (%): "))
    ph = float(input("pH: "))
    rainfall = float(input("Rainfall (mm): "))

    crop, top3 = predict_crop(N, P, K, temperature, humidity, ph, rainfall)
    print("\nRecommended Crop:", crop)
    print("Top 3 probable crops:", top3)

# Uncomment to run interactively:
# predict_crop_interactive()